# Notebook 08 — Consolidated Evaluation & Reproducibility Registry

**Purpose:** consolidate outputs from Notebooks 01–07, check output presence and schema, summarize reported metrics, and create a traceable project status report.

**Important:** this notebook does not retrain models or certify scientific validity. It reads existing output artifacts. Missing artifacts are reported as missing, not silently fabricated. Metrics from different tasks are not directly comparable.


In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
from datetime import datetime, timezone

# Locate project root from common notebook launch locations.
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "ML").is_dir() and (p / "Data").is_dir():
            return p
        if p.name.lower() == "ml" and (p.parent / "Data").is_dir():
            return p.parent
    # If launched inside ML/notebooks, infer project root.
    for p in candidates:
        if p.name.lower() == "notebooks" and p.parent.name.lower() == "ml":
            return p.parent.parent
    raise FileNotFoundError(
        "Could not locate project root. Expected project folders Data/ and ML/. "
        f"Current directory: {start}"
    )

PROJECT_ROOT = find_project_root()
ML_DIR = PROJECT_ROOT / "ML"
OUTPUT_ROOT = ML_DIR / "outputs"
REGISTRY_DIR = OUTPUT_ROOT / "notebook_08"
REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Registry output:", REGISTRY_DIR)


Project root: C:\Users\Amey Ravindra Pandit\Downloads\MAT Risk
Output root: C:\Users\Amey Ravindra Pandit\Downloads\MAT Risk\ML\outputs
Registry output: C:\Users\Amey Ravindra Pandit\Downloads\MAT Risk\ML\outputs\notebook_08


In [2]:
# Expected artifacts per prior notebook.
# These are presence checks, not claims that every artifact exists.
EXPECTED = {
    "01_materials_baseline": {
        "folder": "notebook_01",
        "files": ["run_summary.json", "metrics.csv"],
        "description": "Grouped DS1 baseline comparison",
    },
    "02_materials_feature_engineering": {
        "folder": "notebook_02",
        "files": ["grouped_cv_comparison.csv", "selected_model_results.csv", "run_summary.json"],
        "description": "Grouped CV and selected materials models",
    },
    "03_commodity_forecasting": {
        "folder": "notebook_03",
        "files": ["validation_model_comparison.csv", "heldout_test_comparison.csv", "run_summary.json"],
        "description": "Chronological commodity forecast and ablation",
    },
    "04_infrastructure_condition": {
        "folder": "notebook_04",
        "files": ["run_summary.json"],
        "description": "Infrastructure condition prediction and risk proxy",
    },
    "05_financial_insurance": {
        "folder": "notebook_05",
        "files": ["run_summary.json"],
        "description": "Portfolio exposure and event-severity analysis",
    },
    "06_esg_element_prices": {
        "folder": "notebook_06",
        "files": ["run_summary.json"],
        "description": "Element price coverage and ESG-related indicators",
    },
    "07_inverse_design": {
        "folder": "notebook_07",
        "files": ["run_summary.json"],
        "description": "Cost-constrained ranking of existing material records",
    },
}

# Discover alternate names if notebook outputs used a different filename.
ALIASES = {
    "run_summary.json": ["run_summary.json", "summary.json", "notebook_summary.json"],
    "metrics.csv": ["metrics.csv", "model_metrics.csv", "baseline_metrics.csv"],
    "grouped_cv_comparison.csv": ["grouped_cv_comparison.csv", "cv_comparison.csv"],
    "selected_model_results.csv": ["selected_model_results.csv", "selected_models.csv"],
    "validation_model_comparison.csv": ["validation_model_comparison.csv", "validation_comparison.csv"],
    "heldout_test_comparison.csv": ["heldout_test_comparison.csv", "test_comparison.csv"],
}

def locate_artifact(folder, expected_name):
    base = OUTPUT_ROOT / folder
    names = ALIASES.get(expected_name, [expected_name])
    for name in names:
        candidate = base / name
        if candidate.is_file():
            return candidate
    # Search within notebook output folder if nested structure exists.
    if base.exists():
        for name in names:
            matches = list(base.rglob(name))
            if matches:
                return matches[0]
    return None

inventory_rows = []
for notebook_key, spec in EXPECTED.items():
    for filename in spec["files"]:
        found = locate_artifact(spec["folder"], filename)
        inventory_rows.append({
            "notebook": notebook_key,
            "description": spec["description"],
            "expected_artifact": filename,
            "status": "found" if found else "missing",
            "path": str(found.relative_to(PROJECT_ROOT)) if found else "",
        })

artifact_inventory = pd.DataFrame(inventory_rows)
display(artifact_inventory)
artifact_inventory.to_csv(REGISTRY_DIR / "artifact_inventory.csv", index=False)


,notebook,description,expected_artifact,status,path
0,01_materials_baseline,Initial materials-property baseline,run_summary.json,missing,
1,01_materials_baseline,Initial materials-property baseline,metrics.csv,missing,
2,02_materials_feature_engineering,Grouped CV and selected materials models,grouped_cv_comparison.csv,found,ML\outputs\notebook_02\grouped_cv_comparison.csv
3,02_materials_feature_engineering,Grouped CV and selected materials models,selected_model_results.csv,found,ML\outputs\notebook_02\selected_model_results.csv
4,02_materials_feature_engineering,Grouped CV and selected materials models,run_summary.json,found,ML\outputs\notebook_02\run_summary.json
5,03_commodity_forecasting,Chronological commodity forecast and ablation,validation_model_comparison.csv,found,ML\outputs\notebook_03\validation_model_compar...
6,03_commodity_forecasting,Chronological commodity forecast and ablation,heldout_test_comparison.csv,found,ML\outputs\notebook_03\heldout_test_comparison...
7,03_commodity_forecasting,Chronological commodity forecast and ablation,run_summary.json,found,ML\outputs\notebook_03\run_summary.json
8,04_infrastructure_condition,Infrastructure condition prediction and risk p...,run_summary.json,found,ML\outputs\notebook_04\run_summary.json
9,05_financial_insurance,Portfolio exposure and event-severity analysis,run_summary.json,found,ML\outputs\notebook_05\run_summary.json


In [3]:
# Read summaries when present and preserve their contents rather than inventing metrics.
summary_rows = []
summary_payloads = {}

for notebook_key, spec in EXPECTED.items():
    path = locate_artifact(spec["folder"], "run_summary.json")
    if path is None:
        summary_rows.append({
            "notebook": notebook_key,
            "summary_status": "missing",
            "summary_path": "",
            "top_level_keys": "",
            "notes": "No summary JSON found in expected output folder.",
        })
        continue

    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        summary_payloads[notebook_key] = payload
        summary_rows.append({
            "notebook": notebook_key,
            "summary_status": "readable",
            "summary_path": str(path.relative_to(PROJECT_ROOT)),
            "top_level_keys": ", ".join(map(str, payload.keys())) if isinstance(payload, dict) else "(non-object JSON)",
            "notes": "",
        })
    except Exception as exc:
        summary_rows.append({
            "notebook": notebook_key,
            "summary_status": "invalid_json",
            "summary_path": str(path.relative_to(PROJECT_ROOT)),
            "top_level_keys": "",
            "notes": repr(exc),
        })

summary_inventory = pd.DataFrame(summary_rows)
display(summary_inventory)
summary_inventory.to_csv(REGISTRY_DIR / "summary_inventory.csv", index=False)


,notebook,summary_status,summary_path,top_level_keys,notes
0,01_materials_baseline,missing,,,No summary JSON found in expected output folder.
1,02_materials_feature_engineering,readable,ML\outputs\notebook_02\run_summary.json,"dataset_path, rows_total, rows_formula_parsed,...",
2,03_commodity_forecasting,readable,ML\outputs\notebook_03\run_summary.json,"ds2_path, ds4_path, ds2_rows, ds4_rows, merged...",
3,04_infrastructure_condition,readable,ML\outputs\notebook_04\run_summary.json,"dataset_path, rows, columns, train_rows, test_...",
4,05_financial_insurance,readable,ML\outputs\notebook_05\run_summary.json,"notebook, dataset_paths, dataset_shapes, portf...",
5,06_esg_element_prices,readable,ML\outputs\notebook_06\run_summary.json,"notebook, dataset_rows, formula_parse_success_...",
6,07_inverse_design,readable,ML\outputs\notebook_07\run_summary.json,"ds1_rows, formula_parse_success_rows, elements...",


In [4]:
# Generic metric-table reader. It does not combine incompatible target metrics into one score.
metric_frames = []
metric_candidates = [
    ("01_materials_baseline", "notebook_01", "metrics.csv"),
    ("02_materials_feature_engineering", "notebook_02", "grouped_cv_comparison.csv"),
    ("02_materials_feature_engineering", "notebook_02", "selected_model_results.csv"),
    ("03_commodity_forecasting", "notebook_03", "validation_model_comparison.csv"),
    ("03_commodity_forecasting", "notebook_03", "heldout_test_comparison.csv"),
]
for notebook_key, folder, filename in metric_candidates:
    path = locate_artifact(folder, filename)
    if path is None:
        continue
    try:
        frame = pd.read_csv(path)
        frame.insert(0, "source_notebook", notebook_key)
        frame.insert(1, "source_file", str(path.relative_to(PROJECT_ROOT)))
        metric_frames.append(frame)
    except Exception as exc:
        print(f"Could not read {path}: {exc}")

if metric_frames:
    metrics_registry = pd.concat(metric_frames, ignore_index=True, sort=False)
else:
    metrics_registry = pd.DataFrame(columns=["source_notebook", "source_file"])

display(metrics_registry.head(30))
metrics_registry.to_csv(REGISTRY_DIR / "metrics_registry.csv", index=False)
print("Rows in metric registry:", len(metrics_registry))


,source_notebook,source_file,target,feature_set,model,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_R2_mean,folds,n_train,n_test,MAE,RMSE,R2,Dummy_test_MAE,test_MAE_improvement_pct,n_features,validation_n,directional_accuracy
0,02_materials_feature_engineering,ML\outputs\notebook_02\grouped_cv_comparison.csv,band_gap_eV,composition_only,DummyMean,2.029530,0.017069,2.532320,-0.000202,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02_materials_feature_engineering,ML\outputs\notebook_02\grouped_cv_comparison.csv,band_gap_eV,composition_plus_structure,DummyMean,2.029530,0.017069,2.532320,-0.000202,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,02_materials_feature_engineering,ML\outputs\notebook_02\grouped_cv_comparison.csv,band_gap_eV,composition_only,Ridge,2.034245,0.015846,2.545050,-0.010310,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,02_materials_feature_engineering,ML\outputs\notebook_02\grouped_cv_comparison.csv,band_gap_eV,composition_plus_structure,Ridge,2.039367,0.017701,2.552969,-0.016600,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02_materials_feature_engineering,ML\outputs\notebook_02\grouped_cv_comparison.csv,band_gap_eV,composition_only,ExtraTrees,2.062744,0.013855,2.627255,-0.076824,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,02_materials_feature_engineering,ML\outputs\notebook_02\grouped_cv_comparison.csv,band_gap_eV,composition_only,RandomForest,2.067647,0.011089,2.630144,-0.079029,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,02_materials_feature_engineering,ML\outputs\notebook_02\grouped_cv_comparison.csv,band_gap_eV,composition_plus_structure,RandomForest,2.083675,0.010020,2.638440,-0.085851,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,02_materials_feature_engineering,ML\outputs\notebook_02\grouped_cv_comparison.csv,band_gap_eV,composition_plus_structure,ExtraTrees,2.088383,0.032309,2.655335,-0.099909,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,02_materials_feature_engineering,ML\outputs\notebook_02\grouped_cv_comparison.csv,bulk_modulus_GPa,composition_only,DummyMean,66.363855,0.648917,83.999013,-0.000956,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,02_materials_feature_engineering,ML\outputs\notebook_02\grouped_cv_comparison.csv,bulk_modulus_GPa,composition_plus_structure,DummyMean,66.363855,0.648917,83.999013,-0.000956,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Rows in metric registry: 44


In [5]:
# Explicit limitations and evidence-status register.
LIMITATIONS = [
    {
        "area": "Materials property prediction",
        "status": "experimental",
        "limitation": "Grouped validation results must be reviewed per target; weak baseline performance is not hidden by this registry.",
    },
    {
        "area": "Crystal graph neural network",
        "status": "blocked_by_data",
        "limitation": "DS-1 has no crystal coordinates or atomic structures, so a genuine crystal-structure GNN is not supported by this sample alone.",
    },
    {
        "area": "Commodity forecasting",
        "status": "experimental",
        "limitation": "Chronological test results do not establish profitability; point-in-time provenance of precomputed features must be verified.",
    },
    {
        "area": "Infrastructure survival / RUL",
        "status": "blocked_by_data",
        "limitation": "Cross-sectional bridge snapshot lacks event-time and censoring cohort needed for validated survival analysis.",
    },
    {
        "area": "Financial expected loss",
        "status": "scenario_only",
        "limitation": "Failure-event sample is not a demonstrated representative non-failed portfolio; calibrated PD and actuarial EL are unsupported.",
    },
    {
        "area": "ESG",
        "status": "partial_coverage",
        "limitation": "Commodity-level carbon indicators are not a complete material-specific life-cycle assessment.",
    },
    {
        "area": "Inverse design",
        "status": "screening_only",
        "limitation": "Current approach ranks existing records and does not generate or experimentally validate new materials.",
    },
]
limitations_df = pd.DataFrame(LIMITATIONS)
display(limitations_df)
limitations_df.to_csv(REGISTRY_DIR / "limitations_register.csv", index=False)


,area,status,limitation
0,Materials property prediction,experimental,Grouped validation results must be reviewed pe...
1,Crystal graph neural network,blocked_by_data,DS-1 has no crystal coordinates or atomic stru...
2,Commodity forecasting,experimental,Chronological test results do not establish pr...
3,Infrastructure survival / RUL,blocked_by_data,Cross-sectional bridge snapshot lacks event-ti...
4,Financial expected loss,scenario_only,Failure-event sample is not a demonstrated rep...
5,ESG,partial_coverage,Commodity-level carbon indicators are not a co...
6,Inverse design,screening_only,Current approach ranks existing records and do...


In [6]:
# Final project status report.
found_count = int((artifact_inventory["status"] == "found").sum())
missing_count = int((artifact_inventory["status"] == "missing").sum())
readable_summaries = int((summary_inventory["summary_status"] == "readable").sum())

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "notebook_outputs_root": str(OUTPUT_ROOT),
    "artifact_checks": {
        "expected_artifacts": int(len(artifact_inventory)),
        "found": found_count,
        "missing": missing_count,
    },
    "summary_checks": {
        "notebooks_expected": int(len(EXPECTED)),
        "readable_summaries": readable_summaries,
    },
    "metric_registry_rows": int(len(metrics_registry)),
    "limitations": LIMITATIONS,
    "interpretation": (
        "This is an artifact/reproducibility inventory, not a scientific certification. "
        "Missing files are reported as missing. Metrics are not combined across incompatible tasks."
    ),
    "outputs": [
        "artifact_inventory.csv",
        "summary_inventory.csv",
        "metrics_registry.csv",
        "limitations_register.csv",
        "run_summary.json",
    ],
}
(REGISTRY_DIR / "run_summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))


{
  "generated_at_utc": "2026-10-18T14:33:53.571808+00:00",
  "project_root": "C:\\Users\\Amey Ravindra Pandit\\Downloads\\MAT Risk",
  "notebook_outputs_root": "C:\\Users\\Amey Ravindra Pandit\\Downloads\\MAT Risk\\ML\\outputs",
  "artifact_checks": {
    "expected_artifacts": 12,
    "found": 10,
    "missing": 2
  },
  "summary_checks": {
    "notebooks_expected": 7,
    "readable_summaries": 6
  },
  "metric_registry_rows": 44,
  "limitations": [
    {
      "area": "Materials property prediction",
      "status": "experimental",
      "limitation": "Grouped validation results must be reviewed per target; weak baseline performance is not hidden by this registry."
    },
    {
      "area": "Crystal graph neural network",
      "status": "blocked_by_data",
      "limitation": "DS-1 has no crystal coordinates or atomic structures, so a genuine crystal-structure GNN is not supported by this sample alone."
    },
    {
      "area": "Commodity forecasting",
      "status": "experimenta